# Memvisualisasikan Apa yang Dipelajari Convnet

**Buku:** *Deep Learning dengan Python* — Bab 5, Deep Learning untuk Visi Komputer

Tiga teknik: aktivasi antara, pola filter (gradient ascent di ruang masukan), dan peta panas aktivasi kelas (Grad-CAM).

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Notebook ini **membutuhkan backend TensorFlow** karena memakai `tf.GradientTape` untuk gradien terhadap masukan.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `K.gradients` / `K.function` (dihapus di Keras 3) → `tf.GradientTape`.
- Model dari `bab05_02` dimuat dalam format `.keras`; prapemrosesan `/255` dihapus karena sudah ada lapisan `Rescaling` di dalam model.
- Gambar contoh (gajah) diunduh otomatis; tidak ada path lokal.
- Penumpukan peta panas memakai Matplotlib (tanpa OpenCV).

In [ ]:
import os
# Backend wajib TensorFlow untuk notebook ini
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

## 1. Aktivasi antara
Membutuhkan model `convnet_dengan_augmentasi.keras` hasil `bab05_02`.

In [ ]:
import pathlib
from keras import layers

model = keras.models.load_model("convnet_dengan_augmentasi.keras")
contoh = sorted(pathlib.Path("data/kucing_vs_anjing_kecil/uji/kucing").glob("*.jpg"))[0]
citra = keras.utils.img_to_array(keras.utils.load_img(contoh, target_size=(150, 150)))
tensor_citra = np.expand_dims(citra, axis=0)          # piksel mentah; model punya Rescaling
plt.imshow(citra.astype("uint8")); plt.axis("off"); plt.show()

In [ ]:
lapisan_terpilih = [l for l in model.layers if isinstance(l, (layers.Conv2D, layers.MaxPooling2D))]
model_aktivasi = keras.Model(inputs=model.inputs, outputs=[l.output for l in lapisan_terpilih])
aktivasi = model_aktivasi.predict(tensor_citra, verbose=0)
print([a.shape for a in aktivasi])

plt.matshow(aktivasi[0][0, :, :, 5], cmap="viridis"); plt.title("Kanal ke-5 lapisan pertama"); plt.show()

In [ ]:
per_baris = 16
for lapisan, akt in zip(lapisan_terpilih, aktivasi):
    n_fitur, ukuran = akt.shape[-1], akt.shape[1]
    n_kolom = n_fitur // per_baris
    kisi = np.zeros(((ukuran + 1) * n_kolom - 1, per_baris * (ukuran + 1) - 1))
    for kol in range(n_kolom):
        for brs in range(per_baris):
            kanal = akt[0, :, :, kol * per_baris + brs].copy()
            if kanal.sum() != 0:
                kanal = (kanal - kanal.mean()) / (kanal.std() + 1e-5) * 64 + 128
            kanal = np.clip(kanal, 0, 255)
            kisi[kol * (ukuran + 1):(kol + 1) * ukuran + kol,
                 brs * (ukuran + 1):(brs + 1) * ukuran + brs] = kanal
    skala = 1.0 / ukuran
    plt.figure(figsize=(skala * kisi.shape[1], skala * kisi.shape[0]))
    plt.title(lapisan.name); plt.grid(False); plt.axis("off")
    plt.imshow(kisi, aspect="auto", cmap="viridis")
plt.show()

## 2. Pola filter (gradient ascent di ruang masukan)

In [ ]:
from keras.applications import vgg16

vgg = vgg16.VGG16(weights="imagenet", include_top=False)

_cache_langkah = {}

def _langkah_untuk(nama_lapisan):
    # Satu fungsi gradient ascent (ter-compile) per lapisan, dipakai ulang untuk semua filter.
    if nama_lapisan not in _cache_langkah:
        ekstraktor = keras.Model(vgg.inputs, vgg.get_layer(nama_lapisan).output)

        @tf.function
        def satu_langkah(citra, indeks_filter, laju):
            with tf.GradientTape() as tape:
                tape.watch(citra)
                akt = tf.gather(ekstraktor(citra), indeks_filter, axis=-1)
                loss = tf.reduce_mean(akt[:, 2:-2, 2:-2])
            grad = tape.gradient(loss, citra)
            grad = grad / (tf.sqrt(tf.reduce_mean(tf.square(grad))) + 1e-5)   # normalisasi RMS
            return citra + laju * grad

        _cache_langkah[nama_lapisan] = satu_langkah
    return _cache_langkah[nama_lapisan]

def pola_filter(nama_lapisan, indeks_filter, ukuran=150, langkah=40, laju=1.0):
    satu_langkah = _langkah_untuk(nama_lapisan)
    citra = tf.random.uniform((1, ukuran, ukuran, 3)) * 20 + 128.0
    for _ in range(langkah):
        citra = satu_langkah(citra, tf.constant(indeks_filter), tf.constant(laju))
    return deproses(citra[0].numpy())

def deproses(x):
    x = (x - x.mean()) / (x.std() + 1e-5) * 0.1 + 0.5
    return (np.clip(x, 0, 1) * 255).astype("uint8")

plt.imshow(pola_filter("block3_conv1", 0)); plt.axis("off"); plt.show()

In [ ]:
for nama in ["block1_conv1", "block2_conv1", "block3_conv1", "block4_conv1"]:
    ukuran, margin = 64, 5
    hasil = np.zeros((8 * ukuran + 7 * margin, 8 * ukuran + 7 * margin, 3), dtype="uint8")
    for i in range(8):
        for j in range(8):
            pola = pola_filter(nama, i + j * 8, ukuran=ukuran)
            hasil[i * (ukuran + margin):i * (ukuran + margin) + ukuran,
                  j * (ukuran + margin):j * (ukuran + margin) + ukuran] = pola
    plt.figure(figsize=(10, 10)); plt.title(nama); plt.axis("off"); plt.imshow(hasil)
plt.show()

## 3. Peta panas aktivasi kelas (Grad-CAM)

In [ ]:
vgg_penuh = vgg16.VGG16(weights="imagenet")       # termasuk pengklasifikasi ImageNet
jalur = keras.utils.get_file("elephant.jpg", origin="https://img-datasets.s3.amazonaws.com/elephant.jpg")
citra_asli = keras.utils.img_to_array(keras.utils.load_img(jalur))
x = vgg16.preprocess_input(np.expand_dims(
        keras.utils.img_to_array(keras.utils.load_img(jalur, target_size=(224, 224))), 0))

prediksi = vgg_penuh.predict(x, verbose=0)
print(vgg16.decode_predictions(prediksi, top=3)[0])

In [ ]:
model_grad = keras.Model(vgg_penuh.inputs,
                         [vgg_penuh.get_layer("block5_conv3").output, vgg_penuh.output])
with tf.GradientTape() as tape:
    peta_konv, pred = model_grad(x)
    kelas_teratas = tf.argmax(pred[0])
    skor_kelas = pred[:, kelas_teratas]
grad = tape.gradient(skor_kelas, peta_konv)
bobot_kanal = tf.reduce_mean(grad, axis=(0, 1, 2))              # pentingnya setiap kanal
peta_panas = tf.reduce_sum(peta_konv[0] * bobot_kanal, axis=-1).numpy()
peta_panas = np.maximum(peta_panas, 0) / (peta_panas.max() + 1e-8)
plt.matshow(peta_panas); plt.show()

In [ ]:
import matplotlib as mpl

warna = mpl.colormaps["jet"](np.arange(256))[:, :3]
panas_rgb = warna[np.uint8(255 * peta_panas)]
panas_rgb = keras.utils.array_to_img(panas_rgb).resize((citra_asli.shape[1], citra_asli.shape[0]))
panas_rgb = keras.utils.img_to_array(panas_rgb)
tumpuk = keras.utils.array_to_img(panas_rgb * 0.4 + citra_asli)
tumpuk.save("gajah_gradcam.jpg")
plt.figure(figsize=(8, 6)); plt.imshow(tumpuk); plt.axis("off"); plt.show()